# FSQ-Native Transformer V2
Changes: 32 experts (was 16), 16 levels (was 8)

**Runtime -> Change runtime type -> T4 GPU**

In [ ]:
!pip install transformers torch -q

import os
REPO = '/content/colormlm'
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/Lavender3533/colormlm.git {REPO}

import sys
sys.path.insert(0, REPO)
print('Repo ready')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from colab_fsq_native import FSQTransformer

# V2: bigger codebook
m = FSQTransformer(d=256, nh=4, nl=6, ne=32, nlev=16, ng=16, tk=2, ms=128)
print('V2 params:', m.count_params())

# Quick forward test
ids = torch.randint(0, 100, (2, 32))
lo, dl = m(ids)
print('Forward OK:', lo.shape)

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# More diverse training data
CODE_SAMPLES = [
    'def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)',
    'def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[0]\n    return quicksort([x for x in arr[1:] if x < pivot]) + [pivot] + quicksort([x for x in arr[1:] if x >= pivot])',
    'class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, item):\n        self.items.append(item)\n    def pop(self):\n        return self.items.pop()',
    'class Queue:\n    def __init__(self):\n        self.items = []\n    def enqueue(self, item):\n        self.items.insert(0, item)\n    def dequeue(self):\n        return self.items.pop()',
    'def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1',
    'def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n-1)',
    'def is_palindrome(s):\n    return s == s[::-1]',
    'def reverse_string(s):\n    return s[::-1]',
    'def sum_list(lst):\n    total = 0\n    for x in lst:\n        total += x\n    return total',
    'def max_of_two(a, b):\n    if a > b:\n        return a\n    return b',
    'def is_even(n):\n    return n % 2 == 0',
    'def count_vowels(s):\n    return sum(1 for c in s if c in "aeiou")',
    'def flatten(lst):\n    result = []\n    for item in lst:\n        if isinstance(item, list):\n            result.extend(flatten(item))\n        else:\n            result.append(item)\n    return result',
    'def memoize(func):\n    cache = {}\n    def wrapper(*args):\n        if args not in cache:\n            cache[args] = func(*args)\n        return cache[args]\n    return wrapper',
    'class Node:\n    def __init__(self, val):\n        self.val = val\n        self.next = None',
    'class LinkedList:\n    def __init__(self):\n        self.head = None\n    def append(self, val):\n        node = Node(val)\n        if not self.head:\n            self.head = node\n        else:\n            cur = self.head\n            while cur.next:\n                cur = cur.next\n            cur.next = node',
    'def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr',
    'def insertion_sort(arr):\n    for i in range(1, len(arr)):\n        key = arr[i]\n        j = i - 1\n        while j >= 0 and arr[j] > key:\n            arr[j+1] = arr[j]\n            j -= 1\n        arr[j+1] = key\n    return arr',
    'import os\nimport json\ndef load_config(path):\n    with open(path) as f:\n        return json.load(f)',
    'def retry(func, n=3):\n    for i in range(n):\n        try:\n            return func()\n        except:\n            if i == n - 1:\n                raise',
]

class DS(Dataset):
    def __init__(self, texts, tok, ml=128):
        self.data = []
        for t in texts:
            enc = tok(t, truncation=True, max_length=ml, padding='max_length', return_tensors='pt')
            self.data.append({k: v.squeeze(0) for k, v in enc.items()})
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

print('Loading Qwen 1.5B...')
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', trust_remote_code=True)
teacher = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', torch_dtype=torch.float16, trust_remote_code=True).to(DEVICE)
teacher.eval()

# V2 student: more experts, more levels
student = FSQTransformer(vocab=len(tok), d=256, nh=4, nl=6, ne=32, nlev=16, ng=16, tk=2, ms=128).to(DEVICE)
print('Student params:', student.count_params())

texts = [CODE_SAMPLES[i % len(CODE_SAMPLES)] for i in range(1000)]
ds = DS(texts, tok, 128)
loader = DataLoader(ds, batch_size=8, shuffle=True)
print('Dataset:', len(ds))

opt = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=0.01)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
best = 999

print('\n=== Training V2 ===')
for ep in range(50):
    student.train()
    tl = 0; nb = 0
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE)
        m = batch['attention_mask'].to(DEVICE)
        with torch.no_grad():
            out = teacher(ids, attention_mask=m, output_hidden_states=True)
            th = out.hidden_states[-1].float()
        logits, dl = student(ids, m, th)
        sl = logits[:, :-1, :].contiguous()
        lb = ids[:, 1:].contiguous()
        ll = F.cross_entropy(sl.view(-1, sl.size(-1)), lb.view(-1), ignore_index=tok.pad_token_id or 0)
        loss = 0.5 * dl + 0.5 * ll
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()
        tl += loss.item(); nb += 1
    sch.step()
    avg = tl / max(nb, 1)
    if (ep + 1) % 5 == 0:
        print('Epoch', ep+1, '/50 | Loss:', round(avg, 4))
    if avg < best:
        best = avg
        torch.save(student.state_dict(), 'student_fsq_v2_best.pt')

print('\nDone! Best:', round(best, 4))

In [ ]:
student.eval()
for p in ['def fibonacci', 'class Stack:', 'import os', 'def reverse']:
    enc = tok(p, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        lo, _ = student(enc['input_ids'])
        nx = lo[:, -1, :].argmax(dim=-1)
    print(p, '->', tok.decode(nx))

from google.colab import files
files.download('student_fsq_v2_best.pt')